In [1]:
# PHASE 1: Environment setup
import os
import sys
import gc
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader

# Root setup (edit if needed)
root = Path("/home/jupyter-1nt23cb058/Capstone")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from gnn.model import STPIGNN, LossBreakdown
import gnn.model as gnn_model
import shared.physics_config as phys_cfg

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CFG = {
    "lr": 1e-5,
    "max_epochs": 15,
    "train_stride": 128,
    "val_stride": 24,
    "window": 12,
    "total_nodes": 154902,
}

def save_state(path, model, optimizer, scaler_amp, epoch, step, best_val, loss_total=None):
    payload = {
        "state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler_amp.state_dict(),
        "epoch": int(epoch),
        "step": int(step),
        "best_val_mse": float(best_val),
        "timestamp": time.ctime(),
    }
    if loss_total is not None:
        payload["loss_total"] = float(loss_total)
    torch.save(payload, path)

print(f"Phase 1 complete. Device: {device}")

Phase 1 complete. Device: cuda


In [2]:
 #PHASE 2: Baselines and constants
BASELINE = {
    "test_scaled_mse": 0.00094703699,
    "test_mae_unscaled": 5.080196,
    "test_rmse_unscaled": 6.913161,
}
TARGET_SCALE = 342.9356
print("Phase 2 complete.")

Phase 2 complete.


In [3]:
# PHASE 3: Spatial partitioning and hard alignment
from sklearn.cluster import KMeans

pyg = torch.load(root / "data/processed/graph/topology_graph_pyg_inference.pt", weights_only=False)
edge_index_global = pyg.edge_index.long().cpu()
edge_attr_global = pyg.edge_attr.float().cpu()
train_mask_global = pyg.train_mask.bool().cpu()
num_nodes_global = int(pyg.num_nodes)

node_map_df = pd.read_parquet(root / "data/processed/graph/topology_nodeid_to_index_map.parquet")
nodes_geo_df = pd.read_parquet(root / "data/graphs/bangalore_utm_nodes.parquet")

coords_aligned = node_map_df.merge(
    nodes_geo_df[["osmid", "x", "y"]],
    left_on="node_id",
    right_on="osmid",
    how="inner"
).sort_values("node_index")

coords_np = coords_aligned[["x", "y"]].to_numpy(dtype="float32")
if len(coords_np) != num_nodes_global:
    raise ValueError(f"Coordinate alignment mismatch: {len(coords_np)} vs {num_nodes_global}")

class SpatialPartition:
    def __init__(self, cid, node_ids, edge_index, edge_attr, train_mask):
        self.cid = int(cid)
        self.n_id = torch.as_tensor(node_ids, dtype=torch.long)   # global node indices
        self.edge_index = edge_index.long()                        # local node indices
        self.edge_attr = edge_attr.float()
        self.train_mask = train_mask.bool()
        self.val_mask = None
        self.upwind_edge_mask = torch.zeros(edge_index.shape[1], dtype=torch.bool)
        self.x = None

print(f"Partitioning {num_nodes_global} nodes into 64 clusters...")
kmeans = KMeans(n_clusters=64, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(coords_np)

cluster_data = []
edge_index_np = edge_index_global.numpy()

for cid in range(64):
    n_ids = np.where(cluster_labels == cid)[0].astype(np.int64)
    if len(n_ids) == 0:
        continue

    keep = np.isin(edge_index_np[0], n_ids) & np.isin(edge_index_np[1], n_ids)
    sub_edge = edge_index_np[:, keep]

    local_map = {int(g): i for i, g in enumerate(n_ids.tolist())}
    re_src = np.array([local_map[int(x)] for x in sub_edge[0]], dtype=np.int64)
    re_dst = np.array([local_map[int(x)] for x in sub_edge[1]], dtype=np.int64)

    keep_t = torch.from_numpy(keep)

    cluster_data.append(
        SpatialPartition(
            cid=cid,
            node_ids=n_ids,
            edge_index=torch.tensor(np.stack([re_src, re_dst]), dtype=torch.long),
            edge_attr=edge_attr_global[keep_t],
            train_mask=train_mask_global[torch.as_tensor(n_ids)],
        )
    )

print(f"Phase 3 complete. Clusters: {len(cluster_data)}")

Partitioning 154902 nodes into 64 clusters...
Phase 3 complete. Clusters: 64


In [4]:
# PHASE 4: Feature manifold injection for cluster tensors
master_df = pd.read_parquet(root / "data/processed/graph/master_scaled_checkpoint.parquet")

feature_11 = [
    "pm2_5_scaled", "pm10_scaled", "nitrogen_dioxide_scaled", "sulphur_dioxide_scaled",
    "carbon_monoxide_scaled", "wind_speed_10m_scaled", "wind_direction_10m_scaled",
    "wind_gusts_10m_scaled", "temperature_2m_scaled", "relative_humidity_2m_scaled", "surface_pressure_scaled"
]
for c in feature_11:
    if c not in master_df.columns:
        raise ValueError(f"Missing feature column: {c}")

time_col = "time" if "time" in master_df.columns else ("timestamp" if "timestamp" in master_df.columns else None)
if time_col is None:
    raise ValueError("master_scaled_checkpoint is missing time/timestamp column")
if "node_index" not in master_df.columns:
    raise ValueError("master_scaled_checkpoint is missing node_index")

data_dict = {}
for node_idx, g in tqdm(master_df.groupby("node_index"), desc="Indexing features"):
    vals = g.sort_values(time_col).tail(12)[feature_11].to_numpy(dtype=np.float32)
    if vals.shape[0] > 0:
        t_vals = torch.zeros((12, 11), dtype=torch.float32)
        n = min(12, vals.shape[0])
        t_vals[:n, :] = torch.from_numpy(vals[-n:])
        data_dict[int(node_idx)] = t_vals

total_energy = 0.0
for c in tqdm(cluster_data, desc="Injecting manifold"):
    feat = torch.zeros((len(c.n_id), 12, 16), dtype=torch.float32)  # [N, T, F]
    for i, gid in enumerate(c.n_id.tolist()):
        if int(gid) in data_dict:
            feat[i, :, :11] = data_dict[int(gid)]
    c.x = feat
    total_energy += float(feat.sum().item())

if total_energy == 0.0:
    raise ValueError("Injected manifold is all zeros")

del master_df, data_dict
gc.collect()
print(f"Phase 4 complete. Manifold energy: {total_energy:.2f}")

Indexing features:   0%|          | 0/17 [00:00<?, ?it/s]

Injecting manifold:   0%|          | 0/64 [00:00<?, ?it/s]

Phase 4 complete. Manifold energy: 1898.62


In [5]:
# PHASE 5: Rebuild raw global buffers for temporal loader
df_raw = pd.read_parquet(root / "data/processed/model_input/model_input_node_hourly_features.parquet")
node_map = pd.read_parquet(root / "data/processed/graph/topology_nodeid_to_index_map.parquet")

if "node_index" not in df_raw.columns:
    if "node_id" not in df_raw.columns:
        raise ValueError("model_input must contain node_index or node_id")
    node_to_idx = dict(zip(node_map["node_id"].values, node_map["node_index"].values))
    df_raw["node_index"] = df_raw["node_id"].map(node_to_idx)

ts_col = "timestamp" if "timestamp" in df_raw.columns else ("time" if "time" in df_raw.columns else None)
if ts_col is None:
    raise ValueError("model_input must contain timestamp or time")

df_raw = df_raw.dropna(subset=["node_index", ts_col]).copy()
df_raw["node_index"] = df_raw["node_index"].astype(np.int64)

if "station_pm25" not in df_raw.columns:
    raise ValueError("station_pm25 column missing in model_input")
df_raw["target_scaled"] = pd.to_numeric(df_raw["station_pm25"], errors="coerce").fillna(0.0).astype(np.float32) / TARGET_SCALE

feature_cols_16 = [
    "station_pm10", "station_pm25", "station_no2", "station_so2", "station_co",
    "weather_wind_speed_10m", "weather_wind_direction_10m", "weather_wind_gusts_10m",
    "weather_temperature_2m", "weather_relative_humidity_2m", "weather_surface_pressure",
    "city_nitrogen_dioxide", "city_sulphur_dioxide", "city_pm2_5", "city_pm10", "city_carbon_monoxide",
]
for c in feature_cols_16:
    if c not in df_raw.columns:
        raise ValueError(f"Missing model_input feature: {c}")

all_times = pd.Index(sorted(df_raw[ts_col].unique()))
T_total = len(all_times)
time_codes = pd.Categorical(df_raw[ts_col], categories=all_times, ordered=True).codes.astype(np.int32)

raw_ts_indices = np.ascontiguousarray(time_codes)
raw_node_indices = np.ascontiguousarray(df_raw["node_index"].values.astype(np.int64))
raw_data_values = np.ascontiguousarray(
    df_raw[feature_cols_16 + ["target_scaled"]].to_numpy(dtype=np.float32)
)

sort_order = np.lexsort((raw_node_indices, raw_ts_indices))
raw_data_values = raw_data_values[sort_order]
raw_ts_indices = raw_ts_indices[sort_order]
raw_node_indices = raw_node_indices[sort_order]

time_breaks = np.searchsorted(raw_ts_indices, np.arange(T_total + 1, dtype=np.int32), side="left")

print({
    "raw_data_values": raw_data_values.shape,
    "raw_node_indices": raw_node_indices.shape,
    "time_breaks": time_breaks.shape,
    "T_total": T_total,
})

{'raw_data_values': (1932700, 17), 'raw_node_indices': (1932700,), 'time_breaks': (87851,), 'T_total': 87850}


In [6]:
# PHASE 6: Valid-sample temporal dataset and loaders
class LazyClusterDataset(Dataset):
    def __init__(self, clusters, t0, t1, window=12, stride=128, require_target_overlap=True):
        self.clusters = clusters
        self.window = int(window)
        self.starts = list(range(int(t0), int(t1) - self.window, int(stride)))
        self.require_target_overlap = bool(require_target_overlap)
        self.cluster_global = [np.asarray(c.n_id.cpu().numpy(), dtype=np.int64) for c in self.clusters]

        self.samples = []
        for c_idx, n_ids in enumerate(self.cluster_global):
            for t_start in self.starts:
                any_overlap = False
                target_overlap = False
                for w in range(self.window):
                    t = t_start + w
                    s_ptr = time_breaks[t]
                    e_ptr = time_breaks[t + 1]
                    if e_ptr <= s_ptr:
                        continue
                    t_nodes = raw_node_indices[s_ptr:e_ptr]
                    has = np.isin(t_nodes, n_ids, assume_unique=False).any()
                    if has:
                        any_overlap = True
                        if w == self.window - 1:
                            target_overlap = True
                if any_overlap and ((not self.require_target_overlap) or target_overlap):
                    self.samples.append((c_idx, t_start))

        if len(self.samples) == 0:
            raise RuntimeError("No valid samples found")

        print({
            "starts_total": len(self.starts),
            "clusters": len(self.clusters),
            "valid_samples": len(self.samples),
        })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        c_idx, t0 = self.samples[idx]
        part = self.clusters[c_idx]
        n_ids = self.cluster_global[c_idx]
        num_nodes = n_ids.shape[0]

        x_win = np.zeros((self.window, num_nodes, 16), dtype=np.float32)  # [T, N, F]
        y_win = np.zeros((num_nodes,), dtype=np.float32)                   # [N]

        for w in range(self.window):
            t = t0 + w
            s_ptr = time_breaks[t]
            e_ptr = time_breaks[t + 1]
            if e_ptr <= s_ptr:
                continue

            t_nodes = raw_node_indices[s_ptr:e_ptr]
            t_vals = raw_data_values[s_ptr:e_ptr]  # [K,17]

            member = np.isin(t_nodes, n_ids, assume_unique=False)
            if not np.any(member):
                continue

            g_sel = t_nodes[member]
            v_sel = t_vals[member]

            loc = np.searchsorted(n_ids, g_sel)
            ok = (loc >= 0) & (loc < num_nodes) & (n_ids[loc] == g_sel)
            if not np.any(ok):
                continue

            loc = loc[ok]
            vals = np.nan_to_num(v_sel[ok], nan=0.0, posinf=0.0, neginf=0.0)

            x_win[w, loc, :] = vals[:, :16]
            if w == self.window - 1:
                y_win[loc] = vals[:, -1]

        if np.count_nonzero(x_win) == 0:
            raise RuntimeError(f"Unexpected empty sample idx={idx}, cluster={c_idx}, t0={t0}")

        return torch.from_numpy(x_win), torch.from_numpy(y_win), c_idx

train_ds = LazyClusterDataset(
    cluster_data, t0=0, t1=87178, window=CFG["window"], stride=CFG["train_stride"], require_target_overlap=True
)
val_ds = LazyClusterDataset(
    cluster_data, t0=87178, t1=87514, window=CFG["window"], stride=CFG["val_stride"], require_target_overlap=True
)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

print({"train_samples": len(train_ds), "val_samples": len(val_ds)})

{'starts_total': 681, 'clusters': 64, 'valid_samples': 10215}
{'starts_total': 14, 'clusters': 64, 'valid_samples': 210}
{'train_samples': 10215, 'val_samples': 210}


In [7]:
# PHASE 7: Model setup
edge_dim = None
for c in cluster_data:
    if c.edge_attr.numel() > 0:
        edge_dim = int(c.edge_attr.shape[-1])
        break
if edge_dim is None:
    raise RuntimeError("No cluster has edges; cannot infer edge_dim")

model = STPIGNN(
    node_in_dim=16,
    edge_dim=edge_dim,
    spatial_hidden_dim=96,
    temporal_hidden_dim=96,
    gnn_layers=2,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-2)
scaler_amp = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

print("Phase 7 complete.")

Phase 7 complete.


In [8]:
# PHASE 8: Cluster integrity guard
print("Running cluster integrity checks...")
bad = []
for part in cluster_data:
    n = int(part.x.shape[0]) if part.x is not None else 0
    e = part.edge_index
    ea = part.edge_attr
    tm = part.train_mask
    um = part.upwind_edge_mask

    if n == 0:
        bad.append((part.cid, "empty_x"))
        continue
    if e.dim() != 2 or e.shape[0] != 2:
        bad.append((part.cid, "bad_edge_index_shape"))
        continue
    if e.numel() > 0:
        emn = int(e.min().item())
        emx = int(e.max().item())
        if emn < 0 or emx >= n:
            bad.append((part.cid, f"edge_oob min={emn} max={emx} n={n}"))
    if ea.shape[0] != e.shape[1]:
        bad.append((part.cid, f"edge_attr_mismatch edge_attr={ea.shape[0]} edges={e.shape[1]}"))
    if tm.numel() != n:
        bad.append((part.cid, f"train_mask_mismatch mask={tm.numel()} n={n}"))
    if um.numel() != e.shape[1]:
        bad.append((part.cid, f"upwind_mask_mismatch upwind={um.numel()} edges={e.shape[1]}"))

if bad:
    print(f"Found malformed clusters: {len(bad)}")
    for row in bad[:20]:
        print(row)
else:
    print("All clusters passed integrity checks.")

Running cluster integrity checks...
All clusters passed integrity checks.


In [9]:
# PHASE 9: Build upwind masks from physics logic
from gnn.angular_diffusion import is_downwind

if "coords_np" not in globals():
    raise NameError("coords_np missing; run Cell 3")

# dominant wind direction from raw model_input if available
wind_dir_deg = 85.0
try:
    if "weather_wind_direction_10m" in df_raw.columns:
        wd = pd.to_numeric(df_raw["weather_wind_direction_10m"], errors="coerce").dropna().to_numpy()
        if len(wd) > 0:
            r = np.deg2rad(wd)
            wind_dir_deg = float((np.rad2deg(np.arctan2(np.sin(r).mean(), np.cos(r).mean())) + 360.0) % 360.0)
except Exception:
    pass

print({"wind_dir_deg_used": round(wind_dir_deg, 2)})

total_edges = 0
total_upwind = 0

for part in cluster_data:
    E = int(part.edge_index.shape[1])
    if E == 0:
        part.upwind_edge_mask = torch.zeros(0, dtype=torch.bool)
        continue

    src_local = part.edge_index[0].cpu().numpy()
    dst_local = part.edge_index[1].cpu().numpy()
    global_ids = part.n_id.cpu().numpy()

    src_g = global_ids[src_local]
    dst_g = global_ids[dst_local]

    src_xy = coords_np[src_g]
    dst_xy = coords_np[dst_g]

    dx = dst_xy[:, 0] - src_xy[:, 0]
    dy = dst_xy[:, 1] - src_xy[:, 1]
    bearing = (np.degrees(np.arctan2(dy, dx)) + 360.0) % 360.0
    reverse_bearing = (bearing + 180.0) % 360.0

    upwind = np.array(
        [is_downwind(float(rb), float(wind_dir_deg), cone_deg=75.0) for rb in reverse_bearing],
        dtype=bool,
    )
    part.upwind_edge_mask = torch.from_numpy(upwind)

    total_edges += E
    total_upwind += int(upwind.sum())

print({
    "edges_total": total_edges,
    "upwind_true_total": total_upwind,
    "upwind_ratio": (total_upwind / total_edges) if total_edges else 0.0,
})

{'wind_dir_deg_used': 84.99}
{'edges_total': 417416, 'upwind_true_total': 136569, 'upwind_ratio': 0.3271772045153995}


In [10]:
# PHASE 10: Audit and sanity probes
print("=== BUFFER AUDIT ===")
print("raw_data_values:", raw_data_values.shape)
print("raw_node_indices:", raw_node_indices.shape)
print("time_breaks:", time_breaks.shape)
print("nonzero raw_data_values:", int(np.count_nonzero(raw_data_values)))
print("finite ratio raw_data_values:", float(np.isfinite(raw_data_values).mean()))

print("\n=== CLUSTER AUDIT ===")
sensor_clusters = sum(int(c.train_mask.sum().item()) > 0 for c in cluster_data)
upwind_clusters = sum(int(c.upwind_edge_mask.sum().item()) > 0 for c in cluster_data)
print({"clusters_with_sensors": sensor_clusters, "total_clusters": len(cluster_data)})
print({"clusters_with_upwind_edges": upwind_clusters, "total_clusters": len(cluster_data)})

print("\n=== DATASET AUDIT ===")
print({"train_len": len(train_ds), "val_len": len(val_ds)})

probe = [0, len(train_ds)//4, len(train_ds)//2, (3*len(train_ds))//4, len(train_ds)-1]
for i in probe:
    x, y, cidx = train_ds[int(i)]
    part = cluster_data[int(cidx)]
    print({
        "idx": int(i),
        "cluster": int(cidx),
        "x_shape": tuple(x.shape),
        "y_shape": tuple(y.shape),
        "x_nonzero": int(torch.count_nonzero(x).item()),
        "y_nonzero": int(torch.count_nonzero(y).item()),
        "train_mask_true": int(part.train_mask.sum().item()),
        "upwind_true": int(part.upwind_edge_mask.sum().item()),
    })

print("Phase 10 complete.")

=== BUFFER AUDIT ===
raw_data_values: (1932700, 17)
raw_node_indices: (1932700,)
time_breaks: (87851,)
nonzero raw_data_values: 32771114
finite ratio raw_data_values: 0.8101604278074866

=== CLUSTER AUDIT ===
{'clusters_with_sensors': 16, 'total_clusters': 64}
{'clusters_with_upwind_edges': 64, 'total_clusters': 64}

=== DATASET AUDIT ===
{'train_len': 10215, 'val_len': 210}
{'idx': 0, 'cluster': 6, 'x_shape': (12, 2777, 16), 'y_shape': (2777,), 'x_nonzero': 144, 'y_nonzero': 1, 'train_mask_true': 1, 'upwind_true': 2811}
{'idx': 2553, 'cluster': 25, 'x_shape': (12, 2510, 16), 'y_shape': (2510,), 'x_nonzero': 384, 'y_nonzero': 2, 'train_mask_true': 2, 'upwind_true': 2277}
{'idx': 5107, 'cluster': 35, 'x_shape': (12, 3024, 16), 'y_shape': (3024,), 'x_nonzero': 60, 'y_nonzero': 1, 'train_mask_true': 1, 'upwind_true': 3093}
{'idx': 7661, 'cluster': 43, 'x_shape': (12, 2766, 16), 'y_shape': (2766,), 'x_nonzero': 192, 'y_nonzero': 1, 'train_mask_true': 1, 'upwind_true': 2309}
{'idx': 10214, 

In [11]:
# --- PHASE 10.5: Pre-Audit (11-Channel Engine Readiness) ---
import numpy as np
import pandas as pd

master_path = root / "data/processed/graph/master_scaled_checkpoint.parquet"
audit_df = pd.read_parquet(master_path)

pollutants_11 = [
    "pm2_5_scaled",
    "pm10_scaled",
    "nitrogen_dioxide_scaled",
    "sulphur_dioxide_scaled",
    "carbon_monoxide_scaled",
    "wind_speed_10m_scaled",
    "wind_direction_10m_scaled",
    "wind_gusts_10m_scaled",
    "temperature_2m_scaled",
    "relative_humidity_2m_scaled",
    "surface_pressure_scaled",
]

missing = [c for c in pollutants_11 if c not in audit_df.columns]
if missing:
    raise ValueError(f"Missing required 11-channel columns: {missing}")

summary_rows = []
for c in pollutants_11:
    s = pd.to_numeric(audit_df[c], errors="coerce")
    total = int(s.shape[0])
    finite = int(np.isfinite(s.to_numpy()).sum())
    non_null = int(s.notna().sum())
    non_zero = int((s.fillna(0.0) != 0.0).sum())
    mn = float(np.nanmin(s.to_numpy())) if non_null > 0 else np.nan
    mx = float(np.nanmax(s.to_numpy())) if non_null > 0 else np.nan
    summary_rows.append({
        "channel": c,
        "total_rows": total,
        "finite_ratio": finite / max(1, total),
        "non_null_ratio": non_null / max(1, total),
        "non_zero_ratio": non_zero / max(1, total),
        "min": mn,
        "max": mx,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("channel")
print("=== 11-Channel Audit Summary ===")
display(summary_df)

# Hard guards
low_finite = summary_df[summary_df["finite_ratio"] < 0.95]
all_zero = summary_df[summary_df["non_zero_ratio"] == 0.0]

if len(low_finite) > 0:
    print("WARNING: channels with finite_ratio < 0.95")
    display(low_finite)

if len(all_zero) > 0:
    raise ValueError(f"FATAL: one or more 11 channels are all-zero: {all_zero['channel'].tolist()}")

print("✅ Pre-audit passed: 11-channel engine has required columns and non-zero signal.")

=== 11-Channel Audit Summary ===


,channel,total_rows,finite_ratio,non_null_ratio,non_zero_ratio,min,max
4,carbon_monoxide_scaled,1070203,1.0,1.0,1.0,-1.278903,9.308512
2,nitrogen_dioxide_scaled,1070203,1.0,1.0,1.0,-0.998097,9.053498
1,pm10_scaled,1070203,1.0,1.0,1.0,-1.190016,29.178961
0,pm2_5_scaled,1070203,1.0,1.0,1.0,-1.178473,5.064438
9,relative_humidity_2m_scaled,1070203,1.0,1.0,1.0,-2.611906,1.317256
3,sulphur_dioxide_scaled,1070203,1.0,1.0,1.0,-1.071800,6.974208
10,surface_pressure_scaled,1070203,1.0,1.0,1.0,-2.785728,3.229060
8,temperature_2m_scaled,1070203,1.0,1.0,1.0,-2.525403,3.858463
6,wind_direction_10m_scaled,1070203,1.0,1.0,1.0,-1.991848,1.891764
7,wind_gusts_10m_scaled,1070203,1.0,1.0,1.0,-2.283382,4.398499


✅ Pre-audit passed: 11-channel engine has required columns and non-zero signal.


In [12]:
# Quick training signal audit (100 batches)
import torch, numpy as np

model.eval()
mask_counts, obs_counts, y_stats = [], [], []

for bi, (xb_raw, yb_raw, c_idx_t) in enumerate(train_loader):
    if bi >= 100:
        break
    c_idx = int(c_idx_t[0].item())
    part = cluster_data[c_idx]

    y = yb_raw  # [B,N]
    m = part.train_mask.bool().unsqueeze(0).expand_as(y)  # [B,N]
    obs = (y.abs() > 1e-8)

    mask_counts.append(int(m.sum().item()))
    obs_counts.append(int((m & obs).sum().item()))
    y_stats.append(float(y[m].abs().mean().item()) if m.any() else 0.0)

print({
    "avg_mask_nodes_per_batch": float(np.mean(mask_counts)),
    "avg_observed_mask_nodes_per_batch": float(np.mean(obs_counts)),
    "observed_over_mask_ratio": float(np.sum(obs_counts) / max(1, np.sum(mask_counts))),
    "mean_abs_target_on_mask": float(np.mean(y_stats)),
})

{'avg_mask_nodes_per_batch': 1.5, 'avg_observed_mask_nodes_per_batch': 1.49, 'observed_over_mask_ratio': 0.9933333333333333, 'mean_abs_target_on_mask': 0.20396290253847837}


In [14]:
# --- FIXED Tier-Aware Dataset: guaranteed PHYS sample path ---
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import numpy as np
import torch

class TierAwareClusterDatasetV2(Dataset):
    """
    Sensor clusters:
      - Build windows from raw buffers (supervised path).
    Non-sensor clusters:
      - Build physics-only samples from cluster.x fallback (no raw overlap required).
    """
    def __init__(
        self,
        clusters,
        t0,
        t1,
        window=12,
        stride=128,
        phys_time_subsample=8,   # keep PHYS sample count manageable
    ):
        self.clusters = clusters
        self.window = int(window)
        self.starts = list(range(int(t0), int(t1) - self.window, int(stride)))
        self.cluster_global = [np.asarray(c.n_id.cpu().numpy(), dtype=np.int64) for c in clusters]
        self.cluster_has_sensor = [bool(c.train_mask.any()) for c in clusters]
        self.phys_time_subsample = max(1, int(phys_time_subsample))

        self.samples = []  # (cluster_idx, t_start, mode) where mode in {"station","phys"}

        for c_idx, n_ids in enumerate(self.cluster_global):
            has_sensor = self.cluster_has_sensor[c_idx]

            if has_sensor:
                # supervised samples must have target overlap at window end
                for t_start in self.starts:
                    target_overlap = False
                    t = t_start + self.window - 1
                    s_ptr = time_breaks[t]
                    e_ptr = time_breaks[t + 1]
                    if e_ptr > s_ptr:
                        t_nodes = raw_node_indices[s_ptr:e_ptr]
                        if np.isin(t_nodes, n_ids, assume_unique=False).any():
                            target_overlap = True
                    if target_overlap:
                        self.samples.append((c_idx, t_start, "station"))
            else:
                # physics-only samples: do NOT require raw buffer overlap
                # subsample over time starts to avoid overpowering STATION batches
                for k, t_start in enumerate(self.starts):
                    if (k % self.phys_time_subsample) == 0:
                        self.samples.append((c_idx, t_start, "phys"))

        if len(self.samples) == 0:
            raise RuntimeError("No samples constructed")

        station_n = sum(1 for _, _, m in self.samples if m == "station")
        phys_n = len(self.samples) - station_n
        print({
            "samples_total": len(self.samples),
            "station_samples": station_n,
            "phys_samples": phys_n,
            "phys_ratio": phys_n / max(1, len(self.samples)),
        })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        c_idx, t0, mode = self.samples[idx]
        part = self.clusters[c_idx]
        n_ids = self.cluster_global[c_idx]
        num_nodes = n_ids.shape[0]

        # [T, N, F], [N]
        x_win = np.zeros((self.window, num_nodes, 16), dtype=np.float32)
        y_win = np.zeros((num_nodes,), dtype=np.float32)

        if mode == "station":
            # build from raw buffers
            for w in range(self.window):
                t = t0 + w
                s_ptr = time_breaks[t]
                e_ptr = time_breaks[t + 1]
                if e_ptr <= s_ptr:
                    continue

                t_nodes = raw_node_indices[s_ptr:e_ptr]
                t_vals = raw_data_values[s_ptr:e_ptr]  # [K,17] -> 16 features + target

                member = np.isin(t_nodes, n_ids, assume_unique=False)
                if not np.any(member):
                    continue

                g_sel = t_nodes[member]
                vals = np.nan_to_num(t_vals[member], nan=0.0, posinf=0.0, neginf=0.0)

                loc = np.searchsorted(n_ids, g_sel)
                ok = (loc >= 0) & (loc < num_nodes) & (n_ids[loc] == g_sel)
                if not np.any(ok):
                    continue

                loc = loc[ok]
                vals = vals[ok]

                x_win[w, loc, :] = vals[:, :16]
                if w == self.window - 1:
                    y_win[loc] = vals[:, -1]

            # hard guard only for station mode
            if np.count_nonzero(x_win) == 0:
                raise RuntimeError(f"Empty station sample idx={idx}, cluster={c_idx}, t0={t0}")

        else:
            # PHYS fallback from manifold tensor
            # part.x is [N, T, F] -> need [T, N, F]
            px = part.x
            if px is None:
                raise RuntimeError(f"part.x is missing for phys sample cluster={c_idx}")

            px = px.detach().cpu().numpy()  # [N,T,F]
            px = np.nan_to_num(px, nan=0.0, posinf=0.0, neginf=0.0)
            x_win = np.transpose(px, (1, 0, 2)).astype(np.float32, copy=False)  # [T,N,F]

            # y stays zeros for phys-only mode

        return torch.from_numpy(x_win), torch.from_numpy(y_win), c_idx

# Rebuild datasets
train_ds = TierAwareClusterDatasetV2(
    cluster_data,
    t0=0,
    t1=87178,
    window=12,
    stride=CFG["train_stride"],
    phys_time_subsample=8,
)
val_ds = TierAwareClusterDatasetV2(
    cluster_data,
    t0=87178,
    t1=87514,
    window=12,
    stride=CFG["val_stride"],
    phys_time_subsample=8,
)

# Weighted sampler to enforce mixed tier batches
sensor_cluster_ids = {i for i, c in enumerate(cluster_data) if bool(c.train_mask.any())}
weights = np.ones(len(train_ds.samples), dtype=np.float64)

phys_idx = []
for i, (c_idx, _, mode) in enumerate(train_ds.samples):
    if (mode == "phys") or (c_idx not in sensor_cluster_ids):
        phys_idx.append(i)

if len(phys_idx) == 0:
    raise RuntimeError("PHYS samples are still zero after fallback dataset build")

target_phys_ratio = 0.35
station_count = len(train_ds.samples) - len(phys_idx)
phys_count = len(phys_idx)

w_station = 1.0
w_phys = (target_phys_ratio * station_count) / max(1e-9, (1.0 - target_phys_ratio) * phys_count)
weights[phys_idx] = w_phys

sampler = WeightedRandomSampler(
    weights=torch.as_tensor(weights, dtype=torch.double),
    num_samples=len(weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    sampler=sampler,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

# Final audit
station_n = sum(1 for _, _, m in train_ds.samples if m == "station")
phys_n = sum(1 for _, _, m in train_ds.samples if m == "phys")
print({
    "train_len": len(train_ds),
    "val_len": len(val_ds),
    "station_samples": station_n,
    "phys_samples": phys_n,
    "phys_ratio": phys_n / max(1, len(train_ds)),
    "w_phys": float(w_phys),
})

{'samples_total': 14343, 'station_samples': 10215, 'phys_samples': 4128, 'phys_ratio': 0.28780589834762604}
{'samples_total': 306, 'station_samples': 210, 'phys_samples': 96, 'phys_ratio': 0.3137254901960784}
{'train_len': 14343, 'val_len': 306, 'station_samples': 10215, 'phys_samples': 4128, 'phys_ratio': 0.28780589834762604, 'w_phys': 1.3324575134168155}


In [15]:
# PHASE 11: Full Training Engine (Hierarchical Bars + Tiered + AMP-safe + Tuple-safe)
import os
import time
import gc
import numpy as np
import torch
from tqdm.auto import tqdm
from gnn.model import LossBreakdown

# -------------------------
# Config
# -------------------------
MAX_EPOCHS = int(CFG.get("max_epochs", 15))
SAVE_INTERVAL_SECONDS = 1800
CLEAN_START = True  # set False if you want checkpoint resume behavior

checkpoint_path = "citywide_stpignn_checkpoint_STABLE.pt"
autosave_path = "citywide_stpignn_autosave.pt"
best_path = "citywide_stpignn_best.pt"

amp_enabled = (device.type == "cuda")

# -------------------------
# Helpers
# -------------------------
def save_state(path, epoch, step, best_val, loss_total=None, note=None):
    payload = {
        "state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler_amp.state_dict(),
        "epoch": int(epoch),
        "step": int(step),
        "best_val_mse": float(best_val),
        "timestamp": time.ctime(),
    }
    if loss_total is not None:
        payload["loss_total"] = float(loss_total)
    if note is not None:
        payload["note"] = str(note)
    torch.save(payload, path)

def load_checkpoint(path):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["state_dict"])
    optimizer.load_state_dict(ck["optimizer_state_dict"])
    if "scaler_state_dict" in ck and ck["scaler_state_dict"] is not None:
        scaler_amp.load_state_dict(ck["scaler_state_dict"])
    return ck

def masked_mae_rmse(pred, target, mask):
    if pred.dim() == 1:
        pred = pred.unsqueeze(0)
    if target.dim() == 1:
        target = target.unsqueeze(0)
    if mask.dim() == 1:
        mask = mask.unsqueeze(0).expand_as(pred)

    if not bool(mask.any()):
        z = pred.new_tensor(0.0)
        return z, z

    err = pred[mask] - target[mask]
    mae = err.abs().mean()
    rmse = torch.sqrt((err * err).mean())
    return mae, rmse

def sample_cluster_idx(sample_tuple):
    # Supports (c_idx, t_start) or (c_idx, t_start, mode) or longer tuples
    return int(sample_tuple[0])

# -------------------------
# Init / Resume policy
# -------------------------
if CLEAN_START:
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-6, weight_decay=1e-2)
    scaler_amp = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    start_epoch = 1
    start_step = 0
    best_val_mse = float("inf")
    print("Clean start enabled: optimizer/scaler reset, epoch starts at 1.")
else:
    if "optimizer" not in globals() or optimizer is None:
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-6, weight_decay=1e-2)
    if "scaler_amp" not in globals() or scaler_amp is None:
        scaler_amp = torch.amp.GradScaler("cuda", enabled=amp_enabled)

    start_epoch = 1
    start_step = 0
    best_val_mse = float("inf")
    resume_from = None

    for p in [checkpoint_path, autosave_path, best_path]:
        if os.path.exists(p):
            try:
                ckpt = load_checkpoint(p)
                start_epoch = int(ckpt.get("epoch", 1))
                start_step = int(ckpt.get("step", 0))
                best_val_mse = float(ckpt.get("best_val_mse", float("inf")))
                resume_from = p
                break
            except Exception as e:
                print(f"Skipping unreadable checkpoint {p}: {e}")

    if resume_from is not None:
        print(f"Resumed from {resume_from} | epoch={start_epoch} step={start_step} best={best_val_mse:.6f}")
    else:
        print("No checkpoint found. Starting fresh.")

# Tuple-safe nodes_per_epoch calculation
if "train_ds" in globals() and hasattr(train_ds, "samples"):
    nodes_per_epoch = int(sum(len(cluster_data[sample_cluster_idx(s)].n_id) for s in train_ds.samples))
else:
    nodes_per_epoch = int(CFG.get("total_nodes", 154902)) * max(1, len(train_loader))

print({
    "epochs": MAX_EPOCHS,
    "steps_per_epoch": len(train_loader),
    "nodes_per_epoch": nodes_per_epoch,
    "amp_enabled": amp_enabled,
    "clean_start": CLEAN_START,
})

last_save_time = time.time()

# Stability controls
MAX_BAD_EVENTS_PER_EPOCH = 50
LR_BACKOFF_FACTOR = 0.7
MIN_LR = 1e-6

# -------------------------
# Train / Validate
# -------------------------
try:
    epoch_bar = tqdm(total=(MAX_EPOCHS - start_epoch + 1), desc="Epochs", position=0, leave=True)

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        model.train()
        curr_lambda = float(phys_cfg.PHYSICS_LOSS_LAMBDA) * min(1.0, epoch / 25.0)

        train_total, train_data, train_phys = [], [], []
        train_mae, train_rmse = [], []
        station_batches, phys_batches = 0, 0
        skip_bad, finite_fail, grad_fail = 0, 0, 0
        nodes_seen_epoch = 0

        ROLL_N = 100
        roll_total, roll_data, roll_phys = [], [], []

        batch_bar = tqdm(total=len(train_loader), desc=f"Epoch {epoch} Batches", position=1, leave=False)
        node_bar = tqdm(total=nodes_per_epoch, desc=f"Epoch {epoch} Nodes", position=2, leave=False)

        for i, (xb_raw, yb_raw, c_idx_t) in enumerate(train_loader):
            if epoch == start_epoch and i < start_step:
                batch_bar.update(1)
                continue

            c_idx = int(c_idx_t[0].item())
            part = cluster_data[c_idx]

            xb = torch.nan_to_num(xb_raw.to(device, non_blocking=amp_enabled), nan=0.0, posinf=0.0, neginf=0.0)
            yb = torch.nan_to_num(yb_raw.to(device, non_blocking=amp_enabled), nan=0.0, posinf=0.0, neginf=0.0)
            num_nodes = int(xb.shape[2])

            edge_i_cpu = part.edge_index.long()
            edge_a_cpu = torch.nan_to_num(part.edge_attr.float(), nan=0.0, posinf=0.0, neginf=0.0)

            if edge_i_cpu.numel() > 0:
                e_min = int(edge_i_cpu.min().item())
                e_max = int(edge_i_cpu.max().item())
                if e_min < 0 or e_max >= num_nodes:
                    skip_bad += 1
                    batch_bar.update(1)
                    continue

            if edge_a_cpu.shape[0] != edge_i_cpu.shape[1]:
                skip_bad += 1
                batch_bar.update(1)
                continue

            edge_i = edge_i_cpu.to(device)
            edge_a = edge_a_cpu.to(device)

            base_mask_1d = part.train_mask.to(device=device, dtype=torch.bool)
            if base_mask_1d.numel() != num_nodes:
                skip_bad += 1
                batch_bar.update(1)
                continue

            base_mask = base_mask_1d.unsqueeze(0).expand_as(yb)
            obs_mask = yb.abs() > 1e-8
            sup_mask = base_mask & obs_mask

            u_mask = part.upwind_edge_mask.to(device=device, dtype=torch.bool)
            if u_mask.numel() != edge_i.shape[1]:
                u_mask = torch.zeros(edge_i.shape[1], dtype=torch.bool, device=device)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                pred = model(x_seq=xb, edge_index=edge_i, edge_attr=edge_a)

                if bool(sup_mask.any()):
                    res = gnn_model.compute_total_loss(
                        pred=pred, target=yb, train_mask=sup_mask,
                        edge_index=edge_i, edge_attr=edge_a,
                        upwind_edge_mask=u_mask, physics_lambda=curr_lambda
                    )
                    mode = "STATION"
                    station_batches += 1
                else:
                    phys_p = gnn_model.physics_upwind_penalty(
                        pred=pred, edge_index=edge_i,
                        upwind_edge_mask=u_mask, edge_attr=edge_a
                    )
                    res = LossBreakdown(
                        total=curr_lambda * phys_p,
                        data=pred.new_tensor(0.0),
                        physics=phys_p
                    )
                    mode = "PHYS"
                    phys_batches += 1

            if not torch.isfinite(res.total):
                finite_fail += 1
                batch_bar.update(1)
                if (finite_fail + grad_fail) > MAX_BAD_EVENTS_PER_EPOCH:
                    break
                continue

            t = float(res.total.detach().item())
            d = float(res.data.detach().item())
            p = float(res.physics.detach().item())

            if (not np.isfinite(t)) or (not np.isfinite(d)) or (not np.isfinite(p)):
                finite_fail += 1
                batch_bar.update(1)
                if (finite_fail + grad_fail) > MAX_BAD_EVENTS_PER_EPOCH:
                    break
                continue

            with torch.no_grad():
                mae_t, rmse_t = masked_mae_rmse(pred, yb, sup_mask)
                mae_v = float(mae_t.detach().item())
                rmse_v = float(rmse_t.detach().item())

            grad_norm = 0.0
            if t > 0.0:
                scaler_amp.scale(res.total).backward()
                scaler_amp.unscale_(optimizer)

                bad_grad = False
                for prm in model.parameters():
                    if prm.grad is not None and (not torch.isfinite(prm.grad).all()):
                        bad_grad = True
                        break

                if bad_grad:
                    optimizer.zero_grad(set_to_none=True)
                    grad_fail += 1
                    scaler_amp.update()
                    batch_bar.update(1)
                    if (finite_fail + grad_fail) > MAX_BAD_EVENTS_PER_EPOCH:
                        break
                    continue

                grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.3).item())

                if not np.isfinite(grad_norm):
                    optimizer.zero_grad(set_to_none=True)
                    grad_fail += 1
                    scaler_amp.update()
                    batch_bar.update(1)
                    if (finite_fail + grad_fail) > MAX_BAD_EVENTS_PER_EPOCH:
                        break
                    continue

                scaler_amp.step(optimizer)
                scaler_amp.update()

            lr_now = float(optimizer.param_groups[0]["lr"])

            train_total.append(t)
            train_data.append(d)
            train_phys.append(p)
            train_mae.append(mae_v)
            train_rmse.append(rmse_v)

            roll_total.append(t)
            roll_data.append(d)
            roll_phys.append(p)
            if len(roll_total) > ROLL_N:
                roll_total.pop(0)
                roll_data.pop(0)
                roll_phys.pop(0)

            rT = float(np.nanmean(roll_total)) if roll_total else 0.0
            rD = float(np.nanmean(roll_data)) if roll_data else 0.0
            rP = float(np.nanmean(roll_phys)) if roll_phys else 0.0

            nodes_seen_epoch += num_nodes
            node_pct = 100.0 * nodes_seen_epoch / max(1, nodes_per_epoch)

            batch_bar.set_postfix({
                "M": mode,
                "T": f"{t:.6f}",
                "D": f"{d:.6f}",
                "P": f"{p:.6f}",
                "rT": f"{rT:.6f}",
                "rD": f"{rD:.6f}",
                "rP": f"{rP:.6f}",
                "MAE": f"{mae_v:.6f}",
                "RMSE": f"{rmse_v:.6f}",
                "gN": f"{grad_norm:.4f}",
                "lr": f"{lr_now:.8f}",
                "S/P": f"{station_batches}/{phys_batches}",
                "Skip": skip_bad,
                "NaN": finite_fail,
                "gBad": grad_fail,
            })
            batch_bar.update(1)

            node_step = min(num_nodes, max(0, nodes_per_epoch - node_bar.n))
            if node_step > 0:
                node_bar.update(node_step)
            node_bar.set_postfix({"Node%": f"{node_pct:.1f}", "Seen": nodes_seen_epoch})

            if i % 100 == 0:
                if amp_enabled:
                    torch.cuda.empty_cache()
                gc.collect()

            if (time.time() - last_save_time) > SAVE_INTERVAL_SECONDS:
                save_state(autosave_path, epoch=epoch, step=i, best_val=best_val_mse, loss_total=t, note="autosave")
                last_save_time = time.time()
                tqdm.write(f"[AUTOSAVE] epoch={epoch} step={i}")

        batch_bar.close()
        node_bar.close()

        bad_events = finite_fail + grad_fail
        if bad_events > 20:
            old_lr = optimizer.param_groups[0]["lr"]
            new_lr = max(MIN_LR, old_lr * LR_BACKOFF_FACTOR)
            for g in optimizer.param_groups:
                g["lr"] = new_lr
            tqdm.write(f"[LR_BACKOFF] bad_events={bad_events} lr {old_lr:.8f} -> {new_lr:.8f}")

        # Validation
        model.eval()
        val_total, val_data, val_phys = [], [], []
        val_mae, val_rmse = [], []
        val_nan = 0

        val_bar = tqdm(total=len(val_loader), desc=f"Epoch {epoch} Validation", position=3, leave=False)

        with torch.no_grad():
            for vxb_raw, vyb_raw, vc_idx_t in val_loader:
                vc_idx = int(vc_idx_t[0].item())
                vpart = cluster_data[vc_idx]

                vxb = torch.nan_to_num(vxb_raw.to(device, non_blocking=amp_enabled), nan=0.0, posinf=0.0, neginf=0.0)
                vyb = torch.nan_to_num(vyb_raw.to(device, non_blocking=amp_enabled), nan=0.0, posinf=0.0, neginf=0.0)
                v_num_nodes = int(vxb.shape[2])

                v_edge_i_cpu = vpart.edge_index.long()
                v_edge_a_cpu = torch.nan_to_num(vpart.edge_attr.float(), nan=0.0, posinf=0.0, neginf=0.0)

                if v_edge_i_cpu.numel() > 0:
                    v_min = int(v_edge_i_cpu.min().item())
                    v_max = int(v_edge_i_cpu.max().item())
                    if v_min < 0 or v_max >= v_num_nodes:
                        val_bar.update(1)
                        continue

                if v_edge_a_cpu.shape[0] != v_edge_i_cpu.shape[1]:
                    val_bar.update(1)
                    continue

                v_edge_i = v_edge_i_cpu.to(device)
                v_edge_a = v_edge_a_cpu.to(device)

                v_base_1d = vpart.val_mask if (hasattr(vpart, "val_mask") and vpart.val_mask is not None) else vpart.train_mask
                v_base_1d = v_base_1d.to(device=device, dtype=torch.bool)
                if v_base_1d.numel() != v_num_nodes:
                    val_bar.update(1)
                    continue

                v_base = v_base_1d.unsqueeze(0).expand_as(vyb)
                v_obs = vyb.abs() > 1e-8
                v_sup = v_base & v_obs

                v_u = vpart.upwind_edge_mask.to(device=device, dtype=torch.bool)
                if v_u.numel() != v_edge_i.shape[1]:
                    v_u = torch.zeros(v_edge_i.shape[1], dtype=torch.bool, device=device)

                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    vp = model(x_seq=vxb, edge_index=v_edge_i, edge_attr=v_edge_a)

                    if bool(v_sup.any()):
                        vr = gnn_model.compute_total_loss(
                            pred=vp, target=vyb, train_mask=v_sup,
                            edge_index=v_edge_i, edge_attr=v_edge_a,
                            upwind_edge_mask=v_u, physics_lambda=curr_lambda
                        )
                    else:
                        v_phys = gnn_model.physics_upwind_penalty(
                            pred=vp, edge_index=v_edge_i,
                            upwind_edge_mask=v_u, edge_attr=v_edge_a
                        )
                        vr = LossBreakdown(total=curr_lambda * v_phys, data=vp.new_tensor(0.0), physics=v_phys)

                if not torch.isfinite(vr.total):
                    val_nan += 1
                    val_bar.update(1)
                    continue

                vt = float(vr.total.item())
                vd = float(vr.data.item())
                vpv = float(vr.physics.item())

                if (not np.isfinite(vt)) or (not np.isfinite(vd)) or (not np.isfinite(vpv)):
                    val_nan += 1
                    val_bar.update(1)
                    continue

                mae_t, rmse_t = masked_mae_rmse(vp, vyb, v_sup)

                val_total.append(vt)
                val_data.append(vd)
                val_phys.append(vpv)
                val_mae.append(float(mae_t.item()))
                val_rmse.append(float(rmse_t.item()))

                val_bar.set_postfix({
                    "ValT": f"{(np.nanmean(val_total) if val_total else 0.0):.6f}",
                    "ValNaN": val_nan
                })
                val_bar.update(1)

        val_bar.close()

        tr_t = float(np.nanmean(train_total)) if train_total else 0.0
        tr_d = float(np.nanmean(train_data)) if train_data else 0.0
        tr_p = float(np.nanmean(train_phys)) if train_phys else 0.0
        tr_mae = float(np.nanmean(train_mae)) if train_mae else 0.0
        tr_rmse = float(np.nanmean(train_rmse)) if train_rmse else 0.0

        va_t = float(np.nanmean(val_total)) if val_total else 0.0
        va_d = float(np.nanmean(val_data)) if val_data else 0.0
        va_p = float(np.nanmean(val_phys)) if val_phys else 0.0
        va_mae = float(np.nanmean(val_mae)) if val_mae else 0.0
        va_rmse = float(np.nanmean(val_rmse)) if val_rmse else 0.0

        epoch_bar.set_postfix({
            "TrainT": f"{tr_t:.6f}",
            "ValT": f"{va_t:.6f}",
            "Lam": f"{curr_lambda:.6f}",
            "S/P": f"{station_batches}/{phys_batches}",
        })
        epoch_bar.update(1)

        tqdm.write(
            f"[EPOCH {epoch}] "
            f"Train T/D/P={tr_t:.6f}/{tr_d:.6f}/{tr_p:.6f} "
            f"MAE/RMSE={tr_mae:.6f}/{tr_rmse:.6f} | "
            f"Val T/D/P={va_t:.6f}/{va_d:.6f}/{va_p:.6f} "
            f"MAE/RMSE={va_mae:.6f}/{va_rmse:.6f} | "
            f"lambda={curr_lambda:.6f} | "
            f"S/P={station_batches}/{phys_batches} "
            f"Skip={skip_bad} TrainNaN={finite_fail} GradBad={grad_fail} ValNaN={val_nan}"
        )

        save_state(checkpoint_path, epoch=epoch + 1, step=0, best_val=best_val_mse, loss_total=va_t, note="stable")

        if 0 < va_t < best_val_mse:
            best_val_mse = va_t
            save_state(best_path, epoch=epoch, step=0, best_val=best_val_mse, loss_total=va_t, note="best")
            tqdm.write(f"[BEST] Updated best validation total: {best_val_mse:.6f}")

        start_step = 0

    epoch_bar.close()

except KeyboardInterrupt:
    tqdm.write("[INTERRUPT] Saving emergency checkpoint...")
    save_state(
        checkpoint_path,
        epoch=epoch if "epoch" in locals() else 1,
        step=i if "i" in locals() else 0,
        best_val=best_val_mse,
        note="interrupt",
    )
    tqdm.write("[INTERRUPT] Emergency checkpoint saved.")

except Exception as e:
    tqdm.write(f"[CRASH] {e}")
    save_state(
        checkpoint_path,
        epoch=epoch if "epoch" in locals() else 1,
        step=i if "i" in locals() else 0,
        best_val=best_val_mse,
        note=f"crash: {e}",
    )
    tqdm.write("[CRASH] Recovery checkpoint saved.")
    raise

print("✅ Phase 11 complete.")

Clean start enabled: optimizer/scaler reset, epoch starts at 1.
{'epochs': 15, 'steps_per_epoch': 14343, 'nodes_per_epoch': 37434142, 'amp_enabled': True, 'clean_start': True}


Epochs:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 1 Batches:   0%|          | 0/14343 [00:00<?, ?it/s]

Epoch 1 Nodes:   0%|          | 0/37434142 [00:00<?, ?it/s]

[LR_BACKOFF] bad_events=51 lr 0.00000300 -> 0.00000210


...

Epoch 15 Batches:   0%|          | 0/14343 [00:00<?, ?it/s]

Epoch 15 Nodes:   0%|          | 0/37434142 [00:00<?, ?it/s]

[LR_BACKOFF] bad_events=51 lr 0.00000100 -> 0.00000100


Epoch 15 Validation:   0%|          | 0/306 [00:00<?, ?it/s]

[EPOCH 15] Train T/D/P=0.030200/0.030180/0.000330 MAE/RMSE=0.079903/0.085842 | Val T/D/P=0.019162/0.019162/0.000013 MAE/RMSE=0.055137/0.058177 | lambda=0.060000 | S/P=2550/1415 Skip=0 TrainNaN=51 GradBad=0 ValNaN=0
✅ Phase 11 complete.


In [16]:
# Re-sync environment flags
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = (device.type == "cuda")

# Initialize Loaders (Batch size 1 is best for cluster-based GNNs to avoid memory spikes)
if "train_ds" in globals() and "val_ds" in globals():
    from torch_geometric.loader import DataLoader
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
    print(f"✅ Loaders ready. AMP: {amp_enabled} | Device: {device}")
    print(f"Total Batches: {len(train_loader)}")
else:
    print("❌ train_ds not found. Please run the cell where the Dataset was created.")

✅ Loaders ready. AMP: True | Device: cuda
Total Batches: 14343


In [18]:
import os

# Define your paths
best_path = "citywide_stpignn_best.pt"

if os.path.isfile(best_path):
    size_mb = os.path.getsize(best_path) / (1024 * 1024)
    print(f"✅ Checkpoint found: {best_path}")
    print(f"📦 File Size: {size_mb:.2f} MB")
    
    # Optional: Quick load test
    try:
        tmp_ck = torch.load(best_path, map_location='cpu')
        print(f"🧠 Model State: Verified (Epoch {tmp_ck.get('epoch', 'N/A')})")
        del tmp_ck
    except Exception as e:
        print(f"❌ Checkpoint Corrupted: {e}")
else:
    print(f"⚠️ File NOT found: {best_path}. Check your directory with !ls")

✅ Checkpoint found: citywide_stpignn_best.pt
📦 File Size: 1.16 MB
🧠 Model State: Verified (Epoch 11)


In [19]:
import torch

# Load the verified best checkpoint
checkpoint = torch.load("citywide_stpignn_best.pt", map_location=device)

print("--- 🏆 PHASE 11.1 FINAL METRICS ---")
print(f"Stored at:        {checkpoint.get('timestamp', 'Unknown')}")
print(f"Best Epoch:       {checkpoint.get('epoch', 'N/A')}")
print(f"Validation Total: {checkpoint.get('best_val_mse', 'N/A'):.6f}")

# Extract extra loss metadata if available
if "loss_total" in checkpoint:
    print(f"Final Step Loss:  {checkpoint['loss_total']:.6f}")

if "note" in checkpoint:
    print(f"Checkpoint Note:  {checkpoint['note']}")

# Model structural check
state_dict = checkpoint['state_dict']
print(f"Weight Tensors:   {len(state_dict.keys())} layers found.")

--- 🏆 PHASE 11.1 FINAL METRICS ---
Stored at:        Wed Apr 29 11:47:16 2026
Best Epoch:       11
Validation Total: 0.018886
Final Step Loss:  0.018886
Checkpoint Note:  best
Weight Tensors:   26 layers found.


In [20]:
import os
import shutil
from datetime import datetime

# 1. Create the archive directory if it doesn't exist
save_dir = "notebooks/models/phase_11_1_final"
os.makedirs(save_dir, exist_ok=True)

# 2. Define the source and destination
source_path = "citywide_stpignn_best.pt"
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
archive_name = f"stpignn_mesh_resolution_E11_{timestamp}.pt"
dest_path = os.path.join(save_dir, archive_name)

# 3. Copy the file
if os.path.exists(source_path):
    shutil.copy2(source_path, dest_path)
    print(f"✅ Learning Archived!")
    print(f"📍 Location: {dest_path}")
    print(f"📉 Validated Loss: 0.018886")
else:
    print(f"⚠️ Error: {source_path} not found in current directory.")

# 4. Optional: Save a small text file with the config used for this run
config_log = os.path.join(save_dir, "model_context.txt")
with open(config_log, "w") as f:
    f.write(f"Phase 11.1 Final Model\n")
    f.write(f"Date: {checkpoint.get('timestamp', 'Unknown')}\n")
    f.write(f"Nodes: 1024 (Individual Sensor Resolution)\n")
    f.write(f"Best Val Total: 0.018886\n")
    f.write(f"Max Lambda used: 0.060\n")
    f.write(f"Hardware: ASUS ROG Strix G16 (RTX 4050)\n")
print(f"📝 Context log saved to {config_log}")

✅ Learning Archived!
📍 Location: notebooks/models/phase_11_1_final/stpignn_mesh_resolution_E11_20260429_1205.pt
📉 Validated Loss: 0.018886
📝 Context log saved to notebooks/models/phase_11_1_final/model_context.txt


In [21]:
# Save the entire partitioned manifold so we never have to run Cell 3/4 again
torch.save(cluster_data, root / "data/processed/graph/partitioned_manifold_64_clusters.pt")
print("✅ High-resolution manifold cached to disk.")

✅ High-resolution manifold cached to disk.
